# Logical S and S† — Rotated Surface Code

This notebook presents the dynamical logical phase-gate protocol from
[arXiv:2412.01391](https://arxiv.org/abs/2412.01391). In one modified
syndrome-extraction round, CNOT layers 1–2 first morph the rotated code into
the half-cycle unrotated-code state. The fold-transversal operation is then
applied before CNOT layers 3–4 and the ancilla measurements return the state
to the rotated code. LightStim treats that complete modified round as one
logical $\bar S$-SE or $\bar S^\dagger$-SE round.

The three diagrams below cover the benchmark configurations used by LightStim:

| Configuration | State transformation | Purpose |
|---|---|---|
| Two-way | $\lvert+\rangle_L \xrightarrow{\bar S} \lvert+Y\rangle_L \xrightarrow{\bar S^\dagger} \lvert+\rangle_L \xrightarrow{M_X} +1$ | Full-circuit round trip |
| One-way $\bar S^\dagger$ | $\lvert+Y\rangle_L \xrightarrow{\bar S^\dagger} \lvert+\rangle_L \xrightarrow{M_X} +1$ | Isolated $\bar S^\dagger$-SE round |
| One-way $\bar S$ | $\lvert+Y\rangle_L \xrightarrow{\bar S} \lvert-\rangle_L \xrightarrow{M_X} -1$ | Isolated $\bar S$-SE round |

The notebook builds noiseless circuits so the diagrams show only the protocol
schedule. Supplying `noise_params` to the same builders produces the benchmark
circuits; simulation, decoding, and result data belong in `benchmarks/logical_ops`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.protocols.rotated_logical_s import (
    build_rotated_s_two_way_circuit,
    build_rotated_s_y_injection_circuit,
)

## Configuration

- `DISTANCE = 3` keeps the spatial diagrams compact.
- `TWO_WAY_ROUNDS = 1` places one ordinary SE round before the first
  $\bar S$-SE round, between the two phase-gate rounds, and after the
  $\bar S^\dagger$-SE round.
- `INJECTION_PADDING_ROUNDS = 1` adds one ordinary noiseless SE round before
  and after the selected phase-gate round. The injection builder also performs
  one initial noiseless projection SE round.
- `INJECTION_PROTOCOL` can be changed from `"corner"` to `"middle"`.

The same builders accept larger distances and padding depths.

In [ ]:
DISTANCE = 3
TWO_WAY_ROUNDS = 1
INJECTION_PADDING_ROUNDS = 1
INJECTION_PROTOCOL = "corner"

## 1. Two-way logical S/S† round trip

Initialize $\lvert+\rangle_L$, perform one ordinary SE round, the
$\bar S$-SE round, one ordinary SE round, the $\bar S^\dagger$-SE round,
and one final ordinary SE round before logical-X readout. The logical state
returns to $\lvert+\rangle_L$, so the noiseless result is $M_X=+1$.

In [ ]:
two_way = build_rotated_s_two_way_circuit(
    distance=DISTANCE,
    rounds=TWO_WAY_ROUNDS,
)
two_way.without_noise().diagram("detslice-with-ops-svg")

## 2. One-way S†: injected +Y to deterministic +X

Inject $\lvert+Y\rangle_L$, perform the noiseless projection and padding
SE rounds, apply the $\bar S^\dagger$-SE round, add the final noiseless
padding round, and measure logical X. Since
$\bar S^\dagger\lvert+Y\rangle_L=\lvert+\rangle_L$, the noiseless result
is $M_X=+1$. When noise is supplied, only the $\bar S^\dagger$-SE round is
noisy.

In [ ]:
s_dag_y_to_plus = build_rotated_s_y_injection_circuit(
    distance=DISTANCE,
    gate="S_DAG",
    padding_rounds=INJECTION_PADDING_ROUNDS,
    injection_protocol=INJECTION_PROTOCOL,
)
s_dag_y_to_plus.without_noise().diagram("detslice-with-ops-svg")

## 3. One-way S: injected +Y to deterministic −X

Inject $\lvert+Y\rangle_L$, perform the noiseless projection and padding
SE rounds, apply the $\bar S$-SE round, add the final noiseless padding
round, and measure logical X. Since
$\bar S\lvert+Y\rangle_L=\lvert-\rangle_L$, the noiseless result is
$M_X=-1$. No physical logical-Z correction is required. When noise is
supplied, only the $\bar S$-SE round is noisy.

In [ ]:
s_y_to_minus = build_rotated_s_y_injection_circuit(
    distance=DISTANCE,
    gate="S",
    padding_rounds=INJECTION_PADDING_ROUNDS,
    injection_protocol=INJECTION_PROTOCOL,
)
s_y_to_minus.without_noise().diagram("detslice-with-ops-svg")